# repairs

> what a reading gets wrong, put right before assembly: a table of fixes, unnumbered runs, numbers borrowed from a second reading, chapters cut by a plan

In [ ]:
#| default_exp repairs

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from difflib import SequenceMatcher
from lekhaka.verses import vnum

These work on pages of blocks, `[{page, blocks}]`, before `assemble`, and change them in place. Each change is noted in `probs` for a proofreader. They are what vedicreader's books needed again and again: the fix tables of the Bṛhat Saṃhitā, the Muhūrta Cintāmaṇi and the Nārada Saṃhitā, the numbering runs of the Phaladīpikā, the second readings of the Jātaka Pārijāta, the heading plans of the Grahacintāmaṇi.

## A table of fixes

`apply_fixes` applies rows `(page, item, kind, read, printed, why)`. `item` is a block's index on its page. `kind` is `fix` (replace `read` with `printed` in the block's text), `doubt` (the same, but only where `witness` says another print agrees; else noted and kept), `slip` (the print's own slip, kept and noted), `number` (the block's number is `printed`), `drop` (the block goes), or `missing` (`printed` is a whole block, put in after `item`; -1 puts it first).

In [ ]:
#| export
def apply_fixes(pages, rows, witness:dict=None, probs:list=None) -> list:
    "Pages with a table of fixes applied: `(page, item, kind, read, printed, why)` rows."
    probs = [] if probs is None else probs
    by, adds = {p['page']: p for p in pages}, []
    for pg, i, kind, a, b, why in rows:
        if (p := by.get(pg)) is None: continue
        bl = p['blocks']
        if kind == 'missing': adds.append((pg, i, b, why)); continue
        it = bl[i] if isinstance(i, int) and 0 <= i < len(bl) else None
        if it is None: probs.append(f'{pg}: block {i} is gone from the reading; {kind} {a} → {b} not applied'); continue
        if kind == 'drop': it['drop'] = True; probs.append(f"{pg}: `{(it.get('text') or '')[:30]}` dropped ({why})"); continue
        if kind == 'number': it['n'] = b; probs.append(f'{pg}: verse number read {a}, printed {b}'); continue
        if a not in (it.get('text') or ''): probs.append(f'{pg}: `{a}` not in block {i}; the {kind} to `{b}` not applied'); continue
        w = (witness or {}).get((pg, i, a))
        if kind == 'fix' or (kind == 'doubt' and w == 'agrees'):
            it['text'] = it['text'].replace(a, b, 1)
            probs.append(f'{pg}: read `{a}`, printed `{b}`' + (f' ({why})' if why else '') + (' [a doubt the witness settles]' if kind == 'doubt' else ''))
        elif kind == 'doubt': probs.append(f'{pg}: doubt: `{a}` kept; perhaps `{b}` ({why})')
        elif kind == 'slip': probs.append(f"{pg}: the print's slip, kept: `{a}` ({why})")
    for _, (pg, after, blk, why) in sorted(enumerate(adds), key=lambda x: (-(x[1][1] if isinstance(x[1][1], int) else 999), -x[0])):
        by[pg]['blocks'].insert(after + 1 if isinstance(after, int) and after >= 0 else 0, dict(blk, added=True))
        probs.append(f"{pg}: {blk.get('kind')} {blk.get('n') or ''} missing from the reading, put in: {(blk.get('text') or '')[:40]}")
    for p in pages: p['blocks'] = [b for b in p['blocks'] if not b.get('drop')]
    return pages

In [ ]:
_pg = lambda: [dict(page='p1', blocks=[dict(kind='verse', n=1, text='रामः वनं गच्छति ॥१॥'), dict(kind='verse', n=3, text='सीता ॥३॥'), dict(kind='prose', text='a note')])]
_p = []
_r = apply_fixes(_pg(), [('p1', 0, 'fix', 'गच्छति', 'गच्छत्', 'the print'), ('p1', 1, 'number', 3, 2, ''), ('p1', 2, 'drop', None, None, 'a footnote'),
                         ('p1', 1, 'missing', None, dict(kind='verse', n=3, text='लक्ष्मणः ॥३॥'), 'at the foot'), ('p1', 0, 'doubt', 'रामः', 'राम', 'faint')], probs=_p)
test_eq([(b['n'], b['text']) for b in _r[0]['blocks']], [(1, 'रामः वनं गच्छत् ॥१॥'), (2, 'सीता ॥३॥'), (3, 'लक्ष्मणः ॥३॥')])
test_eq(len(_p), 5); assert any('doubt' in x for x in _p)
test_eq(apply_fixes(_pg(), [('p1', 0, 'doubt', 'रामः', 'राम', '')], witness={('p1', 0, 'रामः'): 'agrees'})[0]['blocks'][0]['text'][:3], 'राम')

## Numbers

`number_runs` numbers a run of unnumbered verses that fits exactly between two numbered ones (a, k verses, then a+k+1); a colophon, or a heading matching `reset`, starts the count again. `borrow_numbers` gives an unnumbered verse the number a second reading prints on the verse most like it, at a similarity of `at` or more, when no verse on the page has it already.

In [ ]:
#| export
def number_runs(pages, probs:list=None, reset:str=r'ध्याय') -> list:
    "Unnumbered verses numbered where they fit exactly between two numbered ones."
    probs = [] if probs is None else probs
    a, run = 0, []
    for p in pages:
        for b in p['blocks']:
            k = b.get('kind')
            if k == 'colophon' or (k == 'heading' and reset and re.search(reset, b.get('text') or '')): a, run = 0, []; continue
            if k != 'verse': continue
            if (n := vnum(b.get('n'))) is None: run.append((p['page'], b)); continue
            if run and n - a - 1 == len(run):
                for j, (pg, x) in enumerate(run, 1): x['n'] = a + j; probs.append(f'{pg}: verse printed without its number taken as {a + j}, between {a} and {n}')
            a, run = n, []
    return pages

def borrow_numbers(pages, other, probs:list=None, at:float=.6) -> list:
    "Unnumbered verses given the number `other`, a second reading of the same pages, prints on the most similar verse."
    probs = [] if probs is None else probs
    theirs = {p['page']: [(n, b.get('text') or '') for b in p['blocks'] if b.get('kind') == 'verse' and (n := vnum(b.get('n'))) is not None] for p in other}
    for p in pages:
        if not (th := theirs.get(p['page'])): continue
        have = {vnum(b.get('n')) for b in p['blocks'] if b.get('kind') == 'verse'}
        for b in p['blocks']:
            if b.get('kind') != 'verse' or b.get('n') is not None: continue
            sc, n = max((SequenceMatcher(None, b.get('text') or '', t).ratio(), n) for n, t in th)
            if sc >= at and n not in have: b['n'] = n; have.add(n); probs.append(f"{p['page']}: verse {n}'s number, lost in this reading, taken from the other ({sc:.2f})")
    return pages

In [ ]:
_ps = [dict(page=0, blocks=[dict(kind='verse', n=1, text='a'), dict(kind='verse', text='b'), dict(kind='verse', text='c'), dict(kind='verse', n=4, text='d'),
                             dict(kind='colophon', text='इति'), dict(kind='verse', text='e'), dict(kind='verse', n=3, text='f')])]
test_eq([b.get('n') for b in number_runs(_ps)[0]['blocks']], [1, 2, 3, 4, None, None, 3])   # after the colophon one verse cannot fill 1..2
_mine = [dict(page=0, blocks=[dict(kind='verse', text='धर्मक्षेत्रे कुरुक्षेत्रे'), dict(kind='verse', n=2, text='दृष्ट्वा तु')])]
_theirs = [dict(page=0, blocks=[dict(kind='verse', n='१', text='धर्मक्षेत्रे कुरुक्षेत्रे समवेता'), dict(kind='verse', n='२', text='दृष्ट्वा तु')])]
test_eq([b['n'] for b in borrow_numbers(_mine, _theirs)[0]['blocks']], [1, 2])

## Chapters by plan

`cut_by_plan` marks where chapters begin when a print's numbering does not say: `plan` lists `(page, item)` pairs, where `item` is a block index or the start of a heading's text. A `start` block goes in before each, and `assemble` closes the chapter there.

In [ ]:
#| export
def cut_by_plan(pages, plan, probs:list=None) -> list:
    "Pages with a chapter-opening heading at each `(page, item)` of `plan`; `item` is an index or a heading's first words."
    probs = [] if probs is None else probs
    by = {p['page']: p for p in pages}
    for pg, item in plan:
        if (p := by.get(pg)) is None: probs.append(f'{pg}: the plan names a page the reading lacks'); continue
        bl = p['blocks']
        i = item if isinstance(item, int) else next((j for j, b in enumerate(bl) if (b.get('text') or '').startswith(item)), None)
        if i is None or not 0 <= i < len(bl): probs.append(f'{pg}: the plan\'s chapter start {item!r} is not on the page'); continue
        bl.insert(i, dict(kind='start'))
    return pages

In [ ]:
from lekhaka.verses import assemble
_pl = [dict(page=0, blocks=[dict(kind='heading', text='கடவுள் வணக்கம்'), dict(kind='verse', n=1, text='x'), dict(kind='heading', text='வாரநயம்'), dict(kind='verse', n=2, text='y')])]
_c, _ = assemble(cut_by_plan(_pl, [(0, 'வாரநயம்')]))
test_eq([(c['heading'], c['colophon'], [v['n'] for v in c['verses']]) for c in _c], [('கடவுள் வணக்கம்', None, [1]), ('வாரநயம்', None, [2])])
_p = []; cut_by_plan(_pl, [(5, 0), (0, 'அது')], _p); test_eq(len(_p), 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()